In [36]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import folium
from shapely import wkt
from shapely.geometry import box
import branca
import branca.colormap as cm
import matplotlib.colors as mcolors
import networkx as nx
from folium.features import DivIcon
from shapely.geometry import LineString
import osmnx as ox
import networkx as nx
import warnings
warnings.filterwarnings("ignore")

In [37]:
#Full Area Bounding Box
north = 47.62425802431359
south = 47.596252480947555
east  = -122.31765250756945
west  = -122.35919327219091

#South Jackson Street Bounding Box
north, south, east, west = 47.60005109850047, 47.59666779633948, -122.32161307888254, -122.32918690097559
bbox = box(west, south, east, north)

In [38]:
df = pd.read_csv(r'C:\Users\desmo\OneDrive\Documents\STARSWalkability\routing\all_scored_edges_filtered_with_ai.csv')
df['geometry'] = df['geometry'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs='EPSG:4326')
filtered_gdf = gdf[gdf.geometry.intersects(bbox)]

In [39]:
geo_light = gpd.read_file(r'C:\Users\desmo\OneDrive\Documents\STARSWalkability\Lightpoles\data\Seattle_City_Light_Poles.geojson')
geo_light = geo_light[geo_light['HasStreetlight'] =='Yes']
geo_light = geo_light[geo_light.geometry.intersects(bbox)]

In [40]:
def get_norm_density(geo_light, filtered_gdf, buf):
    cutoff = 0.04
    newlights = geo_light.copy()
    newlights.to_crs(epsg=3857, inplace=True)  # Ensure CRS is in meters
    streetsegments_buffered = filtered_gdf.copy()
    streetsegments_buffered.to_crs(epsg=3857, inplace=True)  # Ensure CRS is in meters
    streetsegments_buffered['geometry'] = streetsegments_buffered.geometry.buffer(buf) 
    njoined = gpd.sjoin(newlights, streetsegments_buffered, how='inner', predicate='within')

    light_counts = njoined.groupby('index_right').size()
    count = njoined.groupby('OBJECTID').size()
    streetsegments_buffered['light_count'] = streetsegments_buffered.index.map(light_counts).fillna(0).astype(int)
    streetsegments_buffered['ldensity'] = streetsegments_buffered['light_count'] / streetsegments_buffered['length']
    streetsegments_buffered.to_crs(epsg=4326, inplace=True)
    geo_light.to_crs(epsg=4326, inplace=True)

    streetsegments_buffered['ldensity'].clip(upper=cutoff, inplace=True)
    streetsegments_buffered['ldensity'] = streetsegments_buffered['ldensity'] / cutoff

    unmapped_lights = geo_light[geo_light['OBJECTID'].isin(njoined['OBJECTID']) == False]
    mapped_lights = geo_light[geo_light['OBJECTID'].isin(njoined['OBJECTID']) == True]

    return streetsegments_buffered, unmapped_lights, mapped_lights, count

In [41]:
def get_raw_density(geo_light, filtered_gdf, buf):
    newlights = geo_light.copy()
    newlights.to_crs(epsg=3857, inplace=True)  # Ensure CRS is in meters
    streetsegments_buffered = filtered_gdf.copy()
    streetsegments_buffered.to_crs(epsg=3857, inplace=True)  # Ensure CRS is in meters
    streetsegments_buffered['geometry'] = streetsegments_buffered.geometry.buffer(buf) 
    njoined = gpd.sjoin(newlights, streetsegments_buffered, how='inner', predicate='within')

    light_counts = njoined.groupby('index_right').size()
    count = njoined.groupby('OBJECTID').size()
    streetsegments_buffered['light_count'] = streetsegments_buffered.index.map(light_counts).fillna(0).astype(int)
    streetsegments_buffered['ldensity'] = streetsegments_buffered['light_count'] / streetsegments_buffered['length']
    streetsegments_buffered.to_crs(epsg=4326, inplace=True)
    geo_light.to_crs(epsg=4326, inplace=True)

    streetsegments_buffered['ldensity'].clip(upper=1, inplace=True)

    unmapped_lights = geo_light[geo_light['OBJECTID'].isin(njoined['OBJECTID']) == False]
    mapped_lights = geo_light[geo_light['OBJECTID'].isin(njoined['OBJECTID']) == True]

    return streetsegments_buffered, unmapped_lights, mapped_lights, count

In [42]:
streetsegments_buffered, unmapped_lights, mapped_lights, count = get_norm_density(geo_light, gdf, 6)
streetsegments_raw, unmapped_lights_raw, mapped_lights_raw, count_raw = get_raw_density(geo_light, gdf, 6) 
#streetsegments_buffered.geometry = gdf.geometry
streetsegments_raw.geometry = gdf.geometry

In [43]:
def get_info(streetsegments_buffered):
    print('Number of segments per light count:')
    print(streetsegments_buffered['light_count'].value_counts().sort_values(ascending=False))
    print('Light Density Info:')
    print(streetsegments_buffered['ldensity'].sort_values(ascending=False))
    print('Mean:')
    print(streetsegments_buffered['ldensity'].mean())
    print('Median:')
    print(streetsegments_buffered['ldensity'].median())
    print('Standard Deviation:')
    print(streetsegments_buffered['ldensity'].std())
    print('25th Percentile:')
    print(streetsegments_buffered['ldensity'].quantile(0.25))
    print('75th Percentile:')
    print(streetsegments_buffered['ldensity'].quantile(0.75))
    print('Minimum:')
    print(streetsegments_buffered['ldensity'].min())
    print('Maximum:')
    print(streetsegments_buffered['ldensity'].max())


In [44]:
def get_all_info():
    buffer_distances = [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
    info = pd.DataFrame(columns=['buffer', 'mean_density', 'std_density', 'unmapped', 'multi-mapped', 'norm'])
    for buf in buffer_distances:
        norm_streetsegments_buffered, norm_unmapped_lights, norm_mapped_lights, norm_count = get_norm_density(geo_light, gdf, buf)
        streetsegments_buffered, unmapped_lights, mapped_lights, count = get_raw_density(geo_light, gdf, buf)
        info.loc[len(info)] = [buf, norm_streetsegments_buffered['ldensity'].mean(), norm_streetsegments_buffered['ldensity'].std(), geo_light['OBJECTID'].size - count.size, count[count>1].size, True]
        info.loc[len(info)] = [buf, streetsegments_buffered['ldensity'].mean(), streetsegments_buffered['ldensity'].std(), geo_light['OBJECTID'].size - count.size, count[count>1].size, False]
    return info

In [45]:
def plot_density(streetsegments):
    bounds = streetsegments.total_bounds
    centerlat = (bounds[1] + bounds[3]) / 2
    centerlon = (bounds[0] + bounds[2]) / 2
    m = folium.Map(location=[centerlat, centerlon], zoom_start=16)
    folium.GeoJson(bbox, style_function=lambda x: {'fillColor': 'none', 'color': 'black', 'weight': 2}).add_to(m)
    vmin=streetsegments_buffered['ldensity'].min(),
    vmax=streetsegments_buffered['ldensity'].max()

    colormap = branca.colormap.LinearColormap(
            colors=["red", "orange", "yellow", "green"],
            vmin=vmin,
            vmax=vmax,
            caption="(Buffer-mapped) Score"
        )
    colormap.add_to(m)

    def style_function(feature):
        density = feature['properties']['ldensity']
        return {
            'color': colormap(density),
            'weight': 4,
            'opacity': 0.8 
        }

    folium.GeoJson(streetsegments_buffered, name='Street Segments', style_function=style_function, tooltip=folium.GeoJsonTooltip(fields=['ldensity', 'light_count'])).add_to(m)
    for idx, row in unmapped_lights.iterrows():
        lat = row.geometry.y
        lon = row.geometry.x
        folium.CircleMarker(
            location=[lat, lon],
            radius=3,
            color='blue',
            fill=True,
            fill_opacity=0.7,
            popup=f"Light ID: {idx}"
        ).add_to(m)
    return m

In [46]:
plot_density(streetsegments_buffered).save(r'C:\Users\desmo\OneDrive\Documents\STARSWalkability\Lightpoles\data\light_density_map.html')

In [47]:
streetsegments_info = pd.DataFrame()
streetsegments_info['light_count'] = streetsegments_buffered['light_count']
streetsegments_info['ldensity'] = streetsegments_buffered['ldensity']
streetsegments_info.describe()

,light_count,ldensity
count,18972.000000,18972.000000
mean,0.064094,0.029244
std,0.435743,0.167552
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,9.000000,1.000000
